In [ ]:
# --------------------------------------------------------------
#  Toyota 5-class CNN
# --------------------------------------------------------------

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import os

# --------------------------  SEED  ---------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)          # for multi-GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# --------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------  PARAMETERS  ------------------------
BATCH_SIZE = 32
LR         = 1e-3
EPOCHS     = 10 # enough to reach >=85 % on train/dev
DATA_ROOT  = "./toyota_intact_dataset"
SELECTED_CLASSES = ["corolla", "camry", "rav4", "highlander", "prius"]
# --------------------------------------------------------------

# ---------------------  DATA PREP  ---------------------------
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# ---- custom wrapper that keeps only the 5 classes and remaps labels 0-4 ----
class FilteredToyota(Dataset):
    def __init__(self, root, selected, transform=None):
        full = ImageFolder(root)
        self.selected = selected
        self.class_to_idx = {c: i for i, c in enumerate(selected)}
        self.indices = []
        self.old2new = {}
        for cls in selected:
            old_idx = full.class_to_idx[cls]
            self.old2new[old_idx] = self.class_to_idx[cls]
            self.indices.extend([i for i, (_, l) in enumerate(full) if l == old_idx])
        self.full = full
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        img, old_label = self.full[i]
        new_label = self.old2new[old_label]
        if self.transform:
            img = self.transform(img)
        return img, new_label

# load & split
full_set = FilteredToyota(DATA_ROOT, SELECTED_CLASSES)
print(f"Selected samples: {len(full_set)}")

train_len = int(0.8 * len(full_set))
val_len   = int(0.1 * len(full_set))
test_len  = len(full_set) - train_len - val_len

train_idx, val_idx, test_idx = random_split(
    range(len(full_set)),
    [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(SEED)   # reproducible split
)

train_set = Subset(full_set, train_idx)
val_set   = Subset(full_set, val_idx)
test_set  = Subset(full_set, test_idx)

# apply different transforms
train_set = torchvision.datasets.wrapper.WrapperDataset(
    {i: (train_transform(img), label) for i, (img, label) in enumerate(train_set)}
)
# a tiny helper to apply a transform to a Subset
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self): return len(self.subset)
    def __getitem__(self, i):
        img, label = self.subset[i]
        if self.transform: img = self.transform(img)
        return img, label

train_set = TransformSubset(train_set, train_transform)
val_set   = TransformSubset(val_set,   test_transform)
test_set  = TransformSubset(test_set,  test_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
dev_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)
# --------------------------------------------------------------

# --------------------------  MODEL  ---------------------------
class ToyotaCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   64, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128,256, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(256)

        self.pool  = nn.MaxPool2d(2, 2)
        self.drop  = nn.Dropout(0.5)

        self.fc1   = nn.Linear(256*4*4, 512)
        self.bnfc  = nn.BatchNorm1d(512)
        self.fc2   = nn.Linear(512, n_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # 32 → 16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # 16 → 8
        x = self.pool(F.relu(self.bn3(self.conv3(x))))   # 8  → 4
        x = x.view(-1, 256*4*4)
        x = self.drop(x)
        x = F.relu(self.bnfc(self.fc1(x)))
        x = self.drop(x)
        x = self.fc2(x)
        return x
# --------------------------------------------------------------

# ---------------------  TRAINING HELPERS  --------------------
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return 100.0 * correct / total if total else 0.0

def train_one_epoch(model, loader, crit, opt):
    model.train()
    running_loss = correct = total = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out = model(x)
        loss = crit(out, y)
        loss.backward()
        opt.step()

        running_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total   += y.size(0)
    return running_loss / len(loader), 100.0 * correct / total

def validate(model, loader, crit):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = crit(out, y)
            running_loss += loss.item()
    return running_loss / len(loader)
# --------------------------------------------------------------

# --------------------------  RUN  ---------------------------
model = ToyotaCNN(n_classes=5).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

history = {"train_loss": [], "train_acc": [], "dev_loss": [], "dev_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    dev_loss = validate(model, dev_loader, criterion)
    dev_acc  = evaluate(model, dev_loader)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["dev_loss"].append(dev_loss)
    history["dev_acc"].append(dev_acc)

    print(f"[Epoch {epoch:02d}] "
          f"train_loss: {tr_loss:.3f} | train_acc: {tr_acc:5.2f}% | "
          f"dev_loss: {dev_loss:.3f} | dev_acc: {dev_acc:5.2f}%")

# -----------------------  PLOTS  -------------------------
def plot_history(h):
    ep = range(1, len(h["train_loss"]) + 1)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(ep, h["train_loss"], label="train")
    plt.plot(ep, h["dev_loss"],   label="dev")
    plt.title("Loss"); plt.xlabel("Epoch"); plt.legend(); plt.grid()
    plt.subplot(1,2,2)
    plt.plot(ep, h["train_acc"], label="train")
    plt.plot(ep, h["dev_acc"],   label="dev")
    plt.title("Accuracy (%)"); plt.xlabel("Epoch"); plt.legend(); plt.grid()
    plt.tight_layout(); plt.show()

plot_history(history)

# ---------------------  FINAL TEST  ----------------------
test_acc = evaluate(model, test_loader)
print(f"\n=== FINAL TEST ACCURACY: {test_acc:.2f}% ===")
# --------------------------------------------------------------
